# Praktikum 12 – Capstone: Integrierte LLM-Pipeline
**Applied Generative AI – NLP | Sommersemester 2026**

**Lernziele:** Retrieval, PII-Masking, kontextgebundene Antwort und Selbstvalidierung in einer einzigen Pipeline zusammenführen und begründet bewerten.

**Zielprodukt nach 90 Minuten:**
1. Ein funktionierender Mini-Assistent mit nachvollziehbarer Pipeline.
2. Drei bearbeitete Fallbeispiele mit Antwort, Kontextbasis und Validierung.
3. Eine kurze Architektur- und Risikoanalyse als Abschlussartefakt.

In [ ]:
import importlib
import os
import re
import shutil
import subprocess
import sys
import tempfile
import time
from importlib.util import find_spec

IN_COLAB = "google.colab" in sys.modules
MODEL = os.getenv("LLM_MODEL", "qwen3.5:0.8b").strip()
OLLAMA_BASE_URL = os.getenv("OLLAMA_BASE_URL", "http://127.0.0.1:11434").strip()

REQUIRED = {
    "ollama": ("ollama", "0.6.1"),
    "requests": ("requests", "2.33.1"),
}



def run_install(specs):
    commands = []

    if shutil.which("uv") is not None:
        commands.append(["uv", "pip", "install", "--python", sys.executable, *specs])

    commands.append([sys.executable, "-m", "pip", "install", *specs])
    if not IN_COLAB:
        commands.append([sys.executable, "-m", "pip", "install", "--user", *specs])

    last_error = None
    for command in commands:
        try:
            print("$", " ".join(command))
            subprocess.check_call(command)
            return
        except Exception as exc:
            last_error = exc
    raise RuntimeError(f"Installation failed: {specs} ({last_error})")


missing_specs = [
    install_name
    for import_name, (install_name, version) in REQUIRED.items()
    if find_spec(import_name) is None
]
if missing_specs:
    run_install(missing_specs)
    importlib.invalidate_caches()

missing_after = [import_name for import_name in REQUIRED if find_spec(import_name) is None]
if missing_after:
    raise RuntimeError(f"Missing packages after installation: {', '.join(missing_after)}")

import ollama
import requests

DEFAULT_OPTIONS = {
    "num_ctx": 8192,
    "num_predict": 512,
    "temperature": 0.5,
}


def build_options(**overrides):
    options = DEFAULT_OPTIONS.copy()
    for key, value in overrides.items():
        if value is not None:
            options[key] = value
    return options


def is_local_host(url):
    host = url.split("://", 1)[-1].split("/", 1)[0].split(":", 1)[0].lower()
    return host in {"127.0.0.1", "localhost", "0.0.0.0"}


def wait_for_ollama(base_url, timeout=90):
    deadline = time.time() + timeout
    last_error = None
    while time.time() < deadline:
        try:
            response = requests.get(f"{base_url.rstrip('/')}/api/tags", timeout=5)
            response.raise_for_status()
            return response.json()
        except Exception as exc:
            last_error = exc
            time.sleep(2)
    raise RuntimeError(f"Ollama server at {base_url} is not reachable: {last_error}")


if not is_local_host(OLLAMA_BASE_URL):
    raise RuntimeError("This notebook expects a local Ollama service via OLLAMA_BASE_URL.")

if shutil.which("ollama") is None:
    if sys.platform.startswith("win"):
        raise RuntimeError("Ollama fehlt unter Windows. Installiere es über https://ollama.com/download/windows und führe die Setup-Zelle erneut aus.")
    subprocess.check_call(["bash", "-lc", "curl -fsSL https://ollama.com/install.sh | sh"])

try:
    wait_for_ollama(OLLAMA_BASE_URL, timeout=5)
except RuntimeError:
    log_path = os.path.join(tempfile.gettempdir(), "ollama-notebook.log")
    ollama_log = open(log_path, "a", encoding="utf-8")
    popen_kwargs = {"stdout": ollama_log, "stderr": subprocess.STDOUT}
    if os.name != "nt":
        popen_kwargs["start_new_session"] = True
    subprocess.Popen(["ollama", "serve"], **popen_kwargs)
    wait_for_ollama(OLLAMA_BASE_URL, timeout=90)

os.environ["OLLAMA_HOST"] = OLLAMA_BASE_URL
env = dict(os.environ)
subprocess.check_call(["ollama", "pull", MODEL], env=env)
payload = wait_for_ollama(OLLAMA_BASE_URL, timeout=30)
available_models = [item.get("name", "") for item in payload.get("models", []) if isinstance(item, dict)]
model_aliases = {MODEL, f"{MODEL}:latest"}
if not any(name in model_aliases for name in available_models):
    raise RuntimeError(f"Missing local Ollama model: {MODEL}")

print("Runtime:", "Google Colab" if IN_COLAB else "Local/Jupyter")
print("Model:", MODEL)
print("Available local models:", ", ".join(available_models))
print("Systems ready for the capstone session.")


## Teil 1 – Capstone-Fallstudie mit größerem Kontext (50 min)
Wir bauen einen kleinen Assistenten für didaktisch vereinfachte Regeltexte. Es geht nicht um echte Rechtsberatung, sondern um eine nachvollziehbare LLM-Pipeline.

**Pflichtschritte:**
- Maskiere personenbezogene Angaben in der Anfrage.
- Hole passenden Kontext aus mehreren Regeltexten.
- Erzeuge eine kurze Antwort und validiere sie gegen den Kontext.

**Soll-Ergebnis:** pro Anfrage eine Antwort mit Quellenbasis und Validierungslabel.

In [ ]:
LEGAL_CORPUS = [
    {
        "doc_id": "policy_personality",
        "title": "Persönlichkeitsrechte",
        "text": "Personen dürfen ihre Persönlichkeit frei entfalten, solange sie nicht die Rechte anderer verletzen.",
    },
    {
        "doc_id": "policy_body",
        "title": "Schutz von Leben und Gesundheit",
        "text": "Leben, körperliche Unversehrtheit und Gesundheit anderer Personen haben hohen Schutz und setzen der Handlungsfreiheit klare Grenzen.",
    },
    {
        "doc_id": "policy_data",
        "title": "Umgang mit personenbezogenen Daten",
        "text": "Personenbezogene Daten dürfen nur mit klarer Einwilligung oder eindeutiger Rechtsgrundlage weitergegeben werden.",
    },
    {
        "doc_id": "policy_purpose",
        "title": "Zweckbindung",
        "text": "Daten, die für einen bestimmten Zweck erhoben wurden, sollen nicht ohne neue Grundlage für andere Zwecke genutzt werden.",
    },
    {
        "doc_id": "policy_platform",
        "title": "Plattformregeln",
        "text": "Plattformen dürfen Inhalte moderieren, wenn sie gegen Sicherheitsregeln, Datenschutz oder respektvollen Umgang verstossen.",
    },
    {
        "doc_id": "policy_logging",
        "title": "Dokumentation",
        "text": "Kritische Modellentscheidungen und Datenzugriffe sollten protokolliert werden, damit spätere Prüfungen möglich bleiben.",
    },
]

STOPWORDS = {"ich", "bin", "und", "oder", "der", "die", "das", "ein", "eine", "mit", "ohne", "darf", "ist", "zu", "an"}


def tokenize(text):
    return {token for token in re.findall(r"\w+", text.lower()) if token not in STOPWORDS}


def mask_pii(text):
    masked = re.sub(r"\b[A-Z][a-z]+ [A-Z][a-z]+\b", "[NAME]", text)
    masked = re.sub(r"[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}", "[EMAIL]", masked)
    return masked


def retrieve_documents(query, top_k=3):
    query_tokens = tokenize(query)
    scored_docs = []
    for doc in LEGAL_CORPUS:
        doc_tokens = tokenize(f"{doc['title']} {doc['text']}")
        overlap = len(query_tokens & doc_tokens)
        scored_docs.append((overlap, doc))
    scored_docs.sort(key=lambda item: (item[0], len(item[1]["text"])), reverse=True)
    selected_docs = [doc for score, doc in scored_docs[:top_k] if score > 0]
    if len(selected_docs) < top_k:
        selected_docs = [doc for _, doc in scored_docs[:top_k]]
    return selected_docs


class LegalBot:
    def __init__(self, model):
        self.model = model
        self.options = build_options(temperature=0, num_predict=90)

    def _ask(self, prompt, stop=None):
        response = ollama.chat(
            model=self.model,
            messages=[{"role": "user", "content": prompt}],
            think=False,
            options={**self.options, **({"stop": stop} if stop else {})},
            keep_alive="10m",
        )["message"]["content"].strip()
        if not response:
            raise RuntimeError("The model returned an empty response.")
        return response

    def run(self, user_query):
        safe_query = mask_pii(user_query)
        docs = retrieve_documents(safe_query, top_k=3)
        context = "\n\n".join([f"[{doc['doc_id']}] {doc['title']}: {doc['text']}" for doc in docs])

        answer_prompt = (
            "Du bist ein didaktischer Assistent für vereinfachte Regeltexte. "
            "Antworte in 2 bis 3 kurzen Sätzen auf Deutsch. "
            "Nutze ausschliesslich den gegebenen Kontext und formuliere keine echte Rechtsberatung.\n\n"
            f"Kontext:\n{context}\n\nAnfrage: {safe_query}"
        )
        answer = self._ask(answer_prompt, stop=["\n\n"])

        check_prompt = (
            "Prüfe, ob die Antwort nur Informationen aus dem Kontext wiedergibt. "
            "Antworte exakt mit JA oder NEIN.\n\n"
            f"Kontext:\n{context}\n\nAntwort: {answer}"
        )
        validity = self._ask(check_prompt, stop=["\n"])
        if validity not in {"JA", "NEIN"}:
            raise RuntimeError(f"Unexpected validation label: {validity}")

        return {
            "safe_query": safe_query,
            "doc_ids": [doc["doc_id"] for doc in docs],
            "answer": answer,
            "validity": validity,
        }


bot = LegalBot(MODEL)
report = bot.run("Ich bin Max Mustermann. Darf ich Kundendaten ohne Einwilligung an Partnerfirmen weitergeben?")
print(report)

In [ ]:
CAPSTONE_CASES = [
    "Ich bin Max Mustermann. Darf ich Kundendaten ohne Einwilligung an Partnerfirmen weitergeben?",
    "Darf eine Plattform beleidigende Inhalte löschen?",
    "Wir wollen erhobene Kundendaten später für einen neuen Zweck nutzen. Geht das einfach so?",
]

for query in CAPSTONE_CASES:
    result = bot.run(query)
    print(f"Anfrage: {query}")
    print(f"Maskiert: {result['safe_query']}")
    print(f"Dokumente: {', '.join(result['doc_ids'])}")
    print(f"Antwort: {result['answer']}")
    print(f"Validierung: {result['validity']}\n")

## Teil 2 – Architektur- und Risikoanalyse (20 min)
Im zweiten Teil wird aus dem Notebook eine kleine Projektabgabe. Ihr sollt nicht nur etwas ausführen, sondern die Pipeline begründet beschreiben.

**Verbindliche Artefakte:**
1. Ein kurzer Datenfluss mit den Schritten Eingabe, PII-Masking, Retrieval, Antwort und Validierung.
2. Eine Risikoeinschätzung: Wo kann Information Leakage, Halluzination oder Fehlklassifikation auftreten?
3. Eine begründete Verbesserungsidee für die nächste Iteration.

**Leitfragen:**
- Wo entstehen die höchsten fachlichen Risiken?
- Welche Komponente ist aktuell noch am einfachsten angreifbar?
- Welche Messgröße würdest du als Erstes produktiv überwachen?

In [ ]:
pipeline_steps = [
    "1. Nutzeranfrage empfangen",
    "2. Personenbezogene Daten maskieren",
    "3. Passende Regeldokumente per Token-Overlap holen",
    "4. Antwort nur auf Basis des Kontexts erzeugen",
    "5. Antwort mit zweitem Modellaufruf validieren",
]

print("Referenz-Pipeline:")
for step in pipeline_steps:
    print(step)

## Abschluss und Bewertung
**Pflichtabgabe:**
1. Drei bearbeitete Fallbeispiele mit Antwort, Dokument-IDs und Validierungslabel.
2. Eine kurze Architektur-Skizze auf Basis der Referenz-Pipeline.
3. Eine Risikoanalyse mit mindestens zwei konkreten Schwachstellen und einer Verbesserungsidee.

**Bewertungsraster:**
- Funktioniert die Pipeline stabil für mehrere Anfragen?
- Ist die Antwort am Kontext verankert?
- Wurde PII sichtbar maskiert?
- Ist die Risikoanalyse technisch plausibel?

**Ende des Praktikums.**